# AoC 2024 Day 8 — Resonant Collinearity

**Spark — self-join on frequency**

Puzzle: <https://adventofcode.com/2024/day/8>

---

> **On puzzle text and inputs.** Advent of Code is Eric Wastl's work, and he asks that puzzle text and per-user inputs not be redistributed. So this notebook carries a summary in my own words plus the *published* example, and pulls the real input at runtime from a local cache that is gitignored. Read the puzzle at the link above.

> **Part 1 only.** Advent of Code reveals Part Two only after a correct Part One submission, and this day is unsolved — so part 2's text does not exist to work from yet. Submitting the answer below unlocks it.

---

## The puzzle

A map of a city block. Every non-`.` cell is an antenna, labelled with its frequency — a single letter or digit, **case-sensitive**, so `A` and `a` are unrelated.

Take any two antennas on the same frequency. Two *antinodes* form on the line through them, one beyond each antenna, each placed so that one antenna is exactly twice as far away as the other. Antinodes may land on top of an antenna, and they only count when they fall inside the map.

- **Part 1** — count the distinct in-bounds locations holding an antinode.

## The approach

Same move as day 4: **stop treating the map as a map.** `posexplode` turns it into an `(r, c, freq)` relation, and from there the puzzle is one join.

"Every pair of antennas sharing a frequency" is an **equi-join of the antenna relation with itself on `freq`**, minus the rows where a cell pairs with itself. The grouping the puzzle describes never has to be built — the join key *is* the grouping.

The nice detail is using **ordered** pairs rather than combinations. For a pair *(p, q)* the antinode on q's side is `2q − p`; the one on p's side is `2p − q`. Because the self-join emits both `(p, q)` and `(q, p)`, a single expression `(2*r2 - r, 2*c2 - c)` produces both antinodes. An unordered-pairs framing would need two expressions and a `union`.

Everything after that is filtering: keep the antinodes inside the grid, `distinct()` because different pairs can aim at the same cell, `count()`. The join itself is trivially small — 164 antennas across 42 frequencies, at most 4 sharing any one frequency — so the whole thing is a few hundred rows.

## Setup

Connect to the cluster's Spark Connect endpoint and import the solution.

In [ ]:
import sys

sys.path.insert(0, '..')  # so `aoc_spark` resolves when running from notebooks/

from aoc_spark.session import get_spark
from aoc_spark.inputs import get_input
from aoc_spark.y2024 import day08

spark = get_spark('aoc-2024-day08')
print('Spark', spark.version)

## The published example

The same data the test suite asserts on.

In [ ]:
EXAMPLE = '............\n........0...\n.....0......\n.......0....\n....0.......\n......A.....\n............\n............\n........A...\n.........A..\n............\n............\n'

print('part 1:', day08.part1(spark, EXAMPLE), '(expected 14)')

### Pairs in, antinodes out

Each row below is one ordered pair and the single antinode it projects past `over`. The three counts at the bottom are the point: more pair-rows than in-bounds antinodes, and more in-bounds antinodes than distinct locations.

In [ ]:
from pyspark.sql import functions as F

grid = EXAMPLE.strip().splitlines()
height, width = len(grid), len(grid[0])

a = day08.antennas(spark, EXAMPLE)
a.groupBy('freq').agg(
    F.count('*').alias('n'),
    F.sort_array(F.collect_list(F.struct('r', 'c'))).alias('at'),
).orderBy('freq').show(truncate=False)

# The self-join: one row per *ordered* pair sharing a frequency.
b = a.select(F.col('r').alias('r2'), F.col('c').alias('c2'), F.col('freq').alias('freq2'))
pairs = a.join(
    b,
    (F.col('freq') == F.col('freq2'))
    & ((F.col('r') != F.col('r2')) | (F.col('c') != F.col('c2'))),
)

antinodes = pairs.select(
    'freq',
    F.struct('r', 'c').alias('from'),
    F.struct('r2', 'c2').alias('over'),
    (2 * F.col('r2') - F.col('r')).alias('ar'),
    (2 * F.col('c2') - F.col('c')).alias('ac'),
).withColumn(
    'in_bounds', F.col('ar').between(0, height - 1) & F.col('ac').between(0, width - 1)
)
antinodes.orderBy('freq', 'from', 'over').show(12, truncate=False)

print('ordered pairs:      ', pairs.count())
print('antinodes in bounds:', antinodes.filter('in_bounds').count())
print('distinct locations: ', antinodes.filter('in_bounds').select('ar', 'ac').distinct().count())

## The real input

`get_input` is cache-first: local gitignored file → Postgres → adventofcode.com. In practice it hits the local file and never touches the network.

In [ ]:
import time

data = get_input(2024, 8)
print(f'input: {len(data):,} chars, {len(data.splitlines()):,} lines')

started = time.perf_counter()
answer = day08.part1(spark, data)
print(f'part 1: {answer}  ({(time.perf_counter() - started) * 1000:.0f} ms)')

## Cross-check

Days 6+ have no known-good answer, so correctness rests on an independently written plain-Python implementation agreeing with the Spark one. That is evidence, not proof — a shared misreading of the puzzle would survive both.

In [ ]:
from reference_python.y2024 import day08 as reference

cross = reference.part1(data)
print('reference:', cross)
print('agree:    ', cross == answer)

## Notes & gotchas

- `split(s, '')` emits a **trailing empty string**, so `antennas()` filters both `''` and `'.'`. Drop the empty-string filter and every row gains a phantom antenna at column `width`.
- The join excludes self-pairs by comparing **coordinates**, not identity. That is correct only because no two antennas occupy the same cell — true of the input, but it is an assumption.
- `between(0, height - 1)` is **inclusive on both ends**, which is what you want for a 0-indexed grid. `between(0, height)` would admit a phantom row.
- Grid dimensions are read in Python from the raw string, not derived from the DataFrame, so a ragged grid would produce wrong bounds rather than an error.
- `distinct()` before `count()` is load-bearing: separate pairs frequently project onto the same cell. On the published example the in-bounds antinode rows outnumber the 14 distinct locations.
- An antinode sitting **on top of an antenna** still counts — nothing in the pipeline subtracts antenna cells, and the example depends on that (the topmost `A` is also a `0`-frequency antinode).
- Frequencies are case-sensitive: `A` and `a` are different join keys, which is automatic here but is the kind of thing a `lower()` somewhere upstream would destroy.